# Customer Churn: Responsible Baseline Notebook

**Purpose:** educational baseline only. The supplied data has just 12 rows, so metrics are unstable and must not be used to justify real deployment.

Before running: upload `customer-churn-training.csv` into the notebook's working directory. In Google Colab, use the Files panel to upload it, or change `DATA_PATH` to the uploaded file path.


In [11]:
import pandas as pd
import numpy as np

DATA_PATH = "customer-churn-training.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)


Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


,0
customer_id,object
tenure_months,int64
support_tickets,int64
monthly_spend_inr,int64
last_login_days,int64
plan_type,object
churned,int64


## 1. Data quality checks

In [12]:
print("Missing values per column:")
display(df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("Target counts:")
display(df["churned"].value_counts().sort_index())
print("Target proportions:")
display(df["churned"].value_counts(normalize=True).sort_index().round(3))
display(df.describe(include="all").T)


Missing values per column:


,0
customer_id,0
tenure_months,0
support_tickets,0
monthly_spend_inr,0
last_login_days,0
plan_type,0
churned,0


Exact duplicate rows: 0
Duplicate customer IDs: 0
Target counts:


,count
churned,
0,7
1,5


Target proportions:


,proportion
churned,
0,0.583
1,0.417


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,12,12,C001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure_months,12.0,NaN,NaN,NaN,12.166667,9.242524,1.0,5.5,9.5,18.5,30.0
support_tickets,12.0,NaN,NaN,NaN,2.25,1.602555,0.0,1.0,2.0,3.25,5.0
monthly_spend_inr,12.0,NaN,NaN,NaN,915.666667,332.574895,499.0,724.0,899.0,1074.0,1499.0
last_login_days,12.0,NaN,NaN,NaN,10.0,9.095453,1.0,2.75,7.0,14.75,30.0
plan_type,12,3,Basic,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
churned,12.0,NaN,NaN,NaN,0.416667,0.514929,0.0,0.0,0.0,1.0,1.0


### Initial observations
The supplied dataset has 12 rows, no missing values, no exact duplicate rows, and no duplicate customer IDs. There are 7 examples with `churned = 0` and 5 with `churned = 1`. This is far too small to establish reliable performance or fairness. The source does not document data collection, permission, feature timestamps, or the churn prediction horizon.


## 2. Define target and features

In [13]:
TARGET = "churned"
# customer_id is an identifier, so do not use it as a predictive feature.
FEATURES = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days",
    "plan_type",
]
X = df[FEATURES]
y = df[TARGET]
print("Features:", FEATURES)
print("Target counts:", y.value_counts().to_dict())


Features: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']
Target counts: {0: 7, 1: 5}


## 3. Non-ML baseline

In [14]:
# A majority-class baseline predicts the most common class for every row.
majority_class = y.mode().iloc[0]
baseline_accuracy = (y == majority_class).mean()
print("Majority class:", majority_class)
print(f"Resubstitution baseline accuracy (not a held-out score): {baseline_accuracy:.3f}")
print("Important: this is computed on the same tiny dataset and is only a reference, not a fair test estimate.")


Majority class: 0
Resubstitution baseline accuracy (not a held-out score): 0.583
Important: this is computed on the same tiny dataset and is only a reference, not a fair test estimate.


## 4. Illustrative train/test split and model

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

numeric_features = ["tenure_months", "support_tickets", "monthly_spend_inr", "last_login_days"]
categorical_features = ["plan_type"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ]
)

# Stratification preserves the class proportions as much as possible, but the test set is tiny.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(precision_score(y_test, pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 3))
print("F1:", round(f1_score(y_test, pred, zero_division=0), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, pred, labels=[0, 1]))
print(classification_report(y_test, pred, zero_division=0))


Train rows: 9 | Test rows: 3
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix [[TN, FP], [FN, TP]]:
[[2 0]
 [0 1]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## 5. Interpret cautiously

The test set contains only a few rows. One changed prediction can move a metric dramatically. Do not claim that this model is accurate, fair, calibrated, or ready for use from this exercise. The random split is for demonstration; a real project should use a larger, representative, time-aware holdout after confirming that features precede the churn outcome.

## 6. Responsible-use decision

**Decision: do not deploy.** The dataset is too small and its provenance, permissions, sampling process, label definition, and prediction horizon are undocumented. A future validated model could only be considered for optional supportive outreach, with human oversight, monitoring, and a fallback to the existing support process.
